In [0]:
import re
 
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import FeatureUnion, Pipeline
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_sample_weight
 
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
 
# joblib.parallel_backend, NOT sklearn.utils.parallel_backend -- the sklearn
# copy is just a deprecated re-export (removed as of 1.7) that throws a
# FutureWarning. joblib's own version is the stable, canonical one.
from joblib import Parallel, delayed, parallel_backend
from joblibspark import register_spark

In [0]:
register_spark()

In [0]:
RANDOM_STATE = 4092026
N_SPLITS = 5
DATA_PATH = "/Workspace/Users/iamphillipluong@gmail.com/Datasets/exercise-classification/exercises_dataset_CLEAN.csv" 

# Load Datasets

In [0]:
def norm(s):
    s = str(s).lower()
    s = re.sub(r'\(.*?\)', '', s)          # strip equipment notes in parens
    s = re.sub(r'[^a-z0-9 ]', ' ', s)
    return re.sub(r'\s+', ' ', s).strip()
 
 
def load_data(path=DATA_PATH):
    df = pd.read_csv(path)
    df["norm_text"] = df["exercise"].apply(norm)
    return df
 

In [0]:
def make_vectorizer():
    return FeatureUnion([
        ("word", TfidfVectorizer(ngram_range=(1, 2), min_df=1, stop_words="english")),
        ("char", TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), min_df=1)),
    ])
 
 
def build_pipeline(clf):
    return Pipeline([("tfidf", make_vectorizer()), ("clf", clf)])

In [0]:
def build_models():
    return {
        "XGBoost": XGBClassifier(
            n_estimators=400,
            max_depth=5,
            learning_rate=0.05,
            subsample=0.8,
            colsample_bytree=0.8,
            tree_method="hist",
            objective="multi:softprob",
            eval_metric="mlogloss",
            random_state=RANDOM_STATE,
        ),
        "LightGBM": LGBMClassifier(
            n_estimators=400,
            num_leaves=31,
            learning_rate=0.05,
            subsample=0.8,
            colsample_bytree=0.8,
            class_weight="balanced",
            objective="multiclass",
            random_state=RANDOM_STATE,
        ),
    }

In [0]:
def fit_and_score(model_name, train_idx, test_idx, X_text, y, sample_weight):
    clf = build_models()[model_name]
    pipe = build_pipeline(clf)
 
    X_train, X_test = X_text.iloc[train_idx], X_text.iloc[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]
 
    fit_kwargs = {}
    if model_name == "XGBoost":
        fit_kwargs["clf__sample_weight"] = sample_weight[train_idx]
 
    pipe.fit(X_train, y_train, **fit_kwargs)
    preds = pipe.predict(X_test)
 
    return {
        "model": model_name,
        "f1_macro": f1_score(y_test, preds, average="macro"),
        "accuracy": accuracy_score(y_test, preds),
    }

In [0]:
def run_cv(df):
    le = LabelEncoder()
    y = le.fit_transform(df["label"])
    sample_weight = compute_sample_weight("balanced", y)
    X_text = df["norm_text"]
 
    skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
    folds = list(skf.split(X_text, y))
 
    tasks = [
        (model_name, train_idx, test_idx)
        for model_name in build_models()
        for train_idx, test_idx in folds
    ]
 
    with parallel_backend("spark", n_jobs=len(tasks)):
        raw_results = Parallel(n_jobs=len(tasks))(
            delayed(fit_and_score)(model_name, train_idx, test_idx, X_text, y, sample_weight)
            for model_name, train_idx, test_idx in tasks
        )
 
    results_df = pd.DataFrame(raw_results)
    summary = (
        results_df.groupby("model")
        .agg(
            f1_macro_mean=("f1_macro", "mean"),
            f1_macro_std=("f1_macro", "std"),
            accuracy_mean=("accuracy", "mean"),
            accuracy_std=("accuracy", "std"),
        )
        .sort_values("f1_macro_mean", ascending=False)
    )
    return summary, results_df

In [0]:
def main():
    df = load_data()
    print("Label distribution:")
    print(df["label"].value_counts(), "\n")
 
    summary, per_fold = run_cv(df)
    print("per-fold results:")
    print(per_fold.to_string(index=False))
    print("\n=== 5-fold CV summary ===")
    print(summary.to_string())

In [0]:
df = load_data()
print("Label distribution:")
print(df["label"].value_counts(), "\n")

summary, per_fold = run_cv(df)
print("per-fold results:")
print(per_fold.to_string(index=False))
print("\n=== 5-fold CV summary ===")
print(summary.to_string())